# 01 - ESC-50 Data Exploration

Explore the ESC-50 dataset: class distribution, audio properties, and mel-spectrogram visualisations.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd

from src.config import ESC50_META_PATH, ESC50_AUDIO_DIR, SAMPLE_RATE, N_MELS, N_FFT, HOP_LENGTH
from src.dataset import download_esc50, wav_to_mel_spectrogram, normalise_spectrogram

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
# Download ESC-50 if needed
download_esc50()

In [ ]:
# Load metadata
meta = pd.read_csv(ESC50_META_PATH)
print(f"Dataset shape: {meta.shape}")
print(f"Number of classes: {meta['target'].nunique()}")
print(f"Clips per class: {meta.groupby('target').size().unique()}")
print(f"Folds: {sorted(meta['fold'].unique())}")
meta.head(10)

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(14, 6))
class_counts = meta.groupby('category').size().sort_values()
class_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of clips')
ax.set_title('ESC-50 Class Distribution (50 classes, 40 clips each)')
plt.tight_layout()
plt.show()

In [ ]:
# ESC-50 category groups (5 major categories)
categories = {
    'Animals': meta[meta['target'] < 10]['category'].unique().tolist(),
    'Natural soundscapes': meta[(meta['target'] >= 10) & (meta['target'] < 20)]['category'].unique().tolist(),
    'Human (non-speech)': meta[(meta['target'] >= 20) & (meta['target'] < 30)]['category'].unique().tolist(),
    'Interior/domestic': meta[(meta['target'] >= 30) & (meta['target'] < 40)]['category'].unique().tolist(),
    'Exterior/urban': meta[(meta['target'] >= 40) & (meta['target'] < 50)]['category'].unique().tolist(),
}
for group, cats in categories.items():
    print(f"\n{group}: {cats}")

In [ ]:
# Visualise example audio clips and spectrograms
sample_classes = ['dog', 'rain', 'crying_baby', 'clock_tick', 'helicopter']

fig, axes = plt.subplots(len(sample_classes), 2, figsize=(14, 3 * len(sample_classes)))

for i, cls in enumerate(sample_classes):
    row = meta[meta['category'] == cls].iloc[0]
    filepath = ESC50_AUDIO_DIR / row['filename']
    
    y, sr = librosa.load(str(filepath), sr=SAMPLE_RATE)
    
    # Waveform
    axes[i, 0].plot(np.arange(len(y)) / sr, y, color='steelblue', linewidth=0.5)
    axes[i, 0].set_title(f'{cls} - Waveform')
    axes[i, 0].set_xlabel('Time (s)')
    axes[i, 0].set_ylabel('Amplitude')
    
    # Mel spectrogram
    mel = wav_to_mel_spectrogram(str(filepath))
    librosa.display.specshow(mel, sr=sr, hop_length=HOP_LENGTH, x_axis='time',
                            y_axis='mel', ax=axes[i, 1], cmap='viridis')
    axes[i, 1].set_title(f'{cls} - Mel Spectrogram')

plt.tight_layout()
plt.savefig('../results/data_exploration_spectrograms.png', dpi=150)
plt.show()

In [ ]:
# Listen to example clips
for cls in sample_classes:
    row = meta[meta['category'] == cls].iloc[0]
    filepath = ESC50_AUDIO_DIR / row['filename']
    print(f"\n{cls}:")
    ipd.display(ipd.Audio(str(filepath)))

In [ ]:
# Check normalised spectrogram range
sample_file = ESC50_AUDIO_DIR / meta.iloc[0]['filename']
mel = wav_to_mel_spectrogram(str(sample_file))
mel_norm = normalise_spectrogram(mel)

print(f"Raw mel shape: {mel.shape}")
print(f"Raw mel range: [{mel.min():.2f}, {mel.max():.2f}]")
print(f"Normalised range: [{mel_norm.min():.2f}, {mel_norm.max():.2f}]")
print(f"\nThis will be the input shape to our models: (1, {mel.shape[0]}, {mel.shape[1]})")
print(f"(1 channel, {mel.shape[0]} mel bins, {mel.shape[1]} time frames)")